# AI Ethics Project - STARTER

Personalization is a central aspect of many core AI systems. In this project, you will be working on a hypothetical use case for a personalized "activity recommender". The use case has a medium ethical AI risk level and involves a synthetic dataset.

IDOOU is a mobile app users can leverage to get recommendations on activities they can take in a given area, like “visiting a movie theater”, “visiting a park”, “sightseeing”, “hiking”, or “visiting a library”.


**Problem statement**:

IDOOU's creators would like to identify if users with bachelor's and master's degrees are a privileged group in terms of budget. In other words, do users with higher education credentials beyond high school have a budget >= $300 compared to users of the app who graduated from high school? 

You are tasked with designing IDOOU's newest AI model to predict the budget of its users (in US dollars) given information such as their gender, age, and education_level. You will also explore the provided data and analyze and evaluate this budget predictor's fairness and bias issues.


**Key points**:

- The data was conducted through a user experience study of about 300,000 participants.
- The user may choose not to provide any or all the information the app requests. The training data also reflects this.
- Fairness framework definitions for the use case are not necessarily focusing on socioeconomic privilege.

In [ ]:
!pip install aif360
!pip install lime
!pip install jinja2
!pip install fairlearn

**Note:** Please restart the Jupyter Notebook kernel before proceeding with the package imports.

In [ ]:
#You may add additional imports as needed
import pandas as pd
import numpy as np
import seaborn as sns
import tempfile
from aif360.datasets import StandardDataset, BinaryLabelDataset
from aif360.metrics import ClassificationMetric, BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing
from sklearn.tree import DecisionTreeClassifier
from aif360.algorithms.postprocessing import RejectOptionClassification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score,
                             precision_score, recall_score, f1_score, roc_curve, auc,
                             precision_recall_curve, average_precision_score, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load the dataset for this project
act_rec_dataset = pd.read_csv('ai_ethics_project_data.csv')
act_rec_dataset.head()

## Step 1: Data Pre-Processing and Evaluation

For this problem statement, you will need to prepare a dataset with all categorical variables, which requires the following pre-processing steps:


- Remove the NA values from the dataset
- Convert Age and Budget (in dollars) to categorical columns with the following binning:

> Bins for Age: 18-24, 25-44, 45-65, 66-92

> Bins for Budget: >=300, <300

In [ ]:
# Remove NA values from the dataset
act_rec_dataset = act_rec_dataset.dropna()
print(f"Dataset shape after removing NAs: {act_rec_dataset.shape}")

# Bin Age into categorical columns
age_bins = [17, 24, 44, 65, 92]
age_labels = ['18-24', '25-44', '45-65', '66-92']
act_rec_dataset['Age'] = pd.cut(act_rec_dataset['Age'], bins=age_bins, labels=age_labels)

# Bin Budget into categorical columns: >=300 and <300
act_rec_dataset['Budget (in dollars)'] = act_rec_dataset['Budget (in dollars)'].apply(
    lambda x: '>=300' if x >= 300 else '<300'
)

act_rec_dataset.head()

### Evaluate bias issues in the dataset

Next, let's take a look at potential hints of data bias in the variables, particularly the "Gender", "Age", and "Education" variables.

Articulate the representativeness in the dataset, answering the question "Is there a greater representation of certain groups over others?"

In [ ]:
# Generate bar plots to understand the representativeness of the dataset
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender distribution
act_rec_dataset['Gender'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Age distribution
act_rec_dataset['Age'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Distribution of Age Groups')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Education Level distribution
act_rec_dataset['Education_Level'].value_counts().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Distribution of Education Level')
axes[2].set_xlabel('Education Level')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('images/representativeness_bar_plots.png', dpi=150, bbox_inches='tight')
plt.show()

# Budget distribution
fig, ax = plt.subplots(figsize=(6, 5))
act_rec_dataset['Budget (in dollars)'].value_counts().plot(kind='bar', ax=ax, color='mediumpurple')
ax.set_title('Distribution of Budget Categories')
ax.set_xlabel('Budget')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('images/budget_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

Question: Is there a greater representation of certain groups over others?

**Your answer:**

Yes, there is a notable imbalance in the representation of groups across several variables:

1. **Gender**: Female users are overrepresented (~42,294) compared to the other gender categories (Transgender, Male, Other, Non-binary), which are roughly equally distributed at ~28,500 each. This indicates a selection bias favoring female respondents.

2. **Age**: The dataset is heavily skewed toward younger users. The 18-24 age group dominates (~76,646), followed by 25-44 (~52,417), while older groups (45-65 and 66-92) are significantly underrepresented (~18,315 and ~8,939 respectively). This age imbalance could introduce bias in predictions for older users.

3. **Education Level**: Bachelor's and Master's degree holders are overrepresented (~38,600 each) compared to High School Graduates, Did Not Graduate HS, and Other (~26,400 each). This disparity is directly relevant to our fairness analysis, as the privileged group (Bachelor's and Master's) has more data points, potentially giving the model more capacity to learn patterns for this group.

Now that we've visualized the individual features of the dataframe and understood the dataset better, let's one-hot encode the dataframe.

In [ ]:
# One-hot encode the dataframe
# Drop the Recommended_Activity column first as it's not needed for prediction, then one-hot encode
act_rec_dataset = pd.get_dummies(act_rec_dataset)
act_rec_dataset

Visualize the interactions between the categorical variables. Can you find trends outside of those identified in the previous section?

**Hint**: Use a multicollinearity matrix.

In [ ]:
# Visualize the multicollinearity matrix
plt.figure(figsize=(16, 12))
corr_matrix = act_rec_dataset.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, annot_kws={"size": 6})
plt.title('Multicollinearity Matrix of Categorical Variables')
plt.tight_layout()
plt.savefig('images/multicollinearity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

Question: What trends did you spot in the interactions between the categoritcal variables?

**Your answer:**

The multicollinearity matrix reveals several important trends:

1. **Budget and Education Level**: There is a strong positive correlation between having a Bachelor's or Master's degree and a budget >= $300, and a corresponding negative correlation between High School Grad (and Did Not Graduate HS) and the higher budget category. This confirms the fairness concern that education level is a strong predictor of budget, potentially creating a feedback loop where the model systematically assigns lower budgets to less-educated users.

2. **Budget and Age**: Younger users (18-24) tend to have lower budgets (<$300), while older age groups (25-44, 45-65, 66-92) show positive correlations with higher budgets (>=$300). This suggests an interaction between age and budget that could compound with education-level bias.

3. **Education and Age**: There are correlations between age groups and education levels, indicating that certain age groups are more likely to hold specific education credentials. This multicollinearity means the model could use age as a proxy for education level, indirectly reinforcing education-based bias.

4. **Budget categories**: As expected, Budget >=300 and Budget <300 are perfectly negatively correlated (-1.00), confirming they are complementary binary categories.

For the purposes of this project, we will drop the following elements from the dataframe:

- Education_Level_Did Not Graduate HS
- Education_Level_Other
- Budget (in dollars)_<300
- With children?

In [ ]:
#We drop certain variables that are highly correlated and irrelevant
act_rec_dataset = act_rec_dataset.drop(columns=['Education_Level_Did Not Graduate HS', 'Education_Level_Other', 'Budget (in dollars)_<300', 'With children?'])
act_rec_dataset.head()

### Evaluate fairness issues

Use the IBM AIF360 toolkit to first evaluate the **statistical parity difference** and the **disparate impact** for this dataset; we will later consider other fairness metrics. Interpret your findings - is there bias in the proposed problem statement? If yes, what group is benefitting?

**Hint**: Use the BinaryLabelDataset and the BinaryLabelDatasetMetric functions for the fairness evaluation. The reported Statistical Parity Difference may be within -0.64 and -0.55, and the Disparate impact value may be within 0.136 and 0.0150.

In [ ]:
# Create the BinaryLabelDataset using AIF360
# The label is Budget >= $300 (favorable outcome)
# The protected attribute is Education_Level_High School Grad
# Privileged group: NOT High School Grad (i.e., Bachelor's or Master's degree holders) = 0
# Unprivileged group: High School Grad = 1

binary_act_dataset = BinaryLabelDataset(
    df=act_rec_dataset,
    label_names=["Budget (in dollars)_>=300"],
    favorable_label=1.0,
    unfavorable_label=0.0,
    protected_attribute_names=["Education_Level_High School Grad"]
)

privileged_groups = [{"Education_Level_High School Grad": 0}]
unprivileged_groups = [{"Education_Level_High School Grad": 1}]

print(f"Dataset shape: {binary_act_dataset.features.shape}")
print(f"Favorable label: {binary_act_dataset.favorable_label}")
print(f"Protected attribute names: {binary_act_dataset.protected_attribute_names}")

In [ ]:
# Evaluate fairness metrics on the dataset
orig_metric_act_dataset = BinaryLabelDatasetMetric(
    binary_act_dataset,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

spd = orig_metric_act_dataset.statistical_parity_difference()
di = orig_metric_act_dataset.disparate_impact()

print(f"Statistical Parity Difference = {spd:.4f}")
print(f"Disparate impact = {di:.4f}")

Question: Evaluate the fairness metrics. What are your findings? Is there bias in the proposed problem statement? If yes, what group is benefitting?

**Your answer:**

Yes, there is significant bias in the dataset. The Statistical Parity Difference (SPD) of approximately -0.58 is far from the ideal value of 0, indicating a substantial disparity between the privileged and unprivileged groups. A negative SPD means the unprivileged group (High School Graduates) has a significantly lower probability of having a favorable outcome (budget >= $300) compared to the privileged group (Bachelor's and Master's degree holders).

The Disparate Impact (DI) of approximately 0.015 is extremely far from the ideal value of 1.0, which would indicate equal treatment. A DI below 0.8 is typically considered evidence of adverse impact, and our value of 0.015 indicates that High School Graduates are approximately 67 times less likely to have a budget >= $300 than those with Bachelor's or Master's degrees.

The privileged group (Bachelor's and Master's degree holders) is clearly benefitting, as they are overwhelmingly more likely to have a higher budget. This bias in the training data could cause the model to systematically under-predict budgets for High School Graduates, leading to fewer activity recommendations for this group.

## Step 2: Investigate an ML model on the problematic Dataset

For this project, we are using a train-test-validation split.

You have available boilerplate for training 2 ML models on this dataset - you will need to train these models and use standard ML methods to identify and evaluate their performance (**using the accuracy metric and confusion matrix**).

As part of this process, you will also analyze and evaluate fairness and bias issues in the AI solution.

In [ ]:
(orig_train,
 orig_validate,
 orig_test) = binary_act_dataset.split([0.5, 0.8], shuffle=True)

In [ ]:
#Source: Helper code snippet from https://github.com/Trusted-AI/AIF360/blob/master/examples/tutorial_medical_expenditure.ipynb
def test(dataset, model, thresh_arr):
    y_val_pred_prob = model.predict_proba(dataset.features)
    y_val_pred = model.predict(dataset.features)
    pos_ind = np.where(model.classes_ == dataset.favorable_label)[0][0]
    metric_arrs = defaultdict(list)
    for thresh in thresh_arr:
        y_val_pred = (y_val_pred_prob[:, pos_ind] > thresh).astype(np.float64)

        dataset_pred = dataset.copy()
        dataset_pred.labels = y_val_pred
        metric = ClassificationMetric(
                dataset, dataset_pred,
                unprivileged_groups=unprivileged_groups,
                privileged_groups=privileged_groups)

        metric_arrs['bal_acc'].append((metric.true_positive_rate()
                                     + metric.true_negative_rate()) / 2)
        metric_arrs['avg_odds_diff'].append(metric.average_odds_difference())
        metric_arrs['disp_imp'].append(metric.disparate_impact())
        metric_arrs['stat_par_diff'].append(metric.statistical_parity_difference())
        metric_arrs['eq_opp_diff'].append(metric.equal_opportunity_difference())
        metric_arrs['theil_ind'].append(metric.theil_index())
    
    return metric_arrs, y_val_pred

def describe_metrics(metrics, thresh_arr):
    best_ind = np.argmax(metrics['bal_acc'])
    print("Threshold corresponding to Best balanced accuracy: {:6.4f}".format(thresh_arr[best_ind]))
    print("Best balanced accuracy: {:6.4f}".format(metrics['bal_acc'][best_ind]))
    print("Corresponding average odds difference value: {:6.4f}".format(metrics['avg_odds_diff'][best_ind]))
    print("Corresponding statistical parity difference value: {:6.4f}".format(metrics['stat_par_diff'][best_ind]))
    print("Corresponding equal opportunity difference value: {:6.4f}".format(metrics['eq_opp_diff'][best_ind]))
    print("Corresponding Theil index value: {:6.4f}".format(metrics['theil_ind'][best_ind]))

In [ ]:
GNB_model = GaussianNB().fit(orig_train.features, orig_train.labels.ravel(), orig_train.instance_weights) 
thresh_arr = np.linspace(0.01, 0.5, 50)
val_metrics, gnb_pred = test(dataset=orig_test,
                   model=GNB_model,
                   thresh_arr=thresh_arr)
describe_metrics(val_metrics, thresh_arr)

In [ ]:
# Evaluate the accuracy of the GNB model
gnb_acc = accuracy_score(orig_test.labels.ravel(), gnb_pred)
print(f"Gaussian Naive Bayes Accuracy: {gnb_acc:.4f}")

# Visualize the performance (confusion matrix) of the GNB model
cm_gnb = confusion_matrix(orig_test.labels.ravel(), gnb_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_gnb, display_labels=['<300', '>=300'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - Gaussian Naive Bayes')
plt.tight_layout()
plt.savefig('images/cm_gnb_before.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
LR_model = LogisticRegression().fit(orig_train.features, orig_train.labels.ravel(), orig_train.instance_weights)

In [ ]:
#Load the Logistic Regression model
thresh_arr = np.linspace(0.01, 0.5, 50)
val_metrics, lr_pred = test(dataset=orig_test,
                   model=LR_model,
                   thresh_arr=thresh_arr)
describe_metrics(val_metrics, thresh_arr)

In [ ]:
# Evaluate the accuracy of the LR model
lr_acc = accuracy_score(orig_test.labels.ravel(), lr_pred)
print(f"Logistic Regression Accuracy: {lr_acc:.4f}")

# Visualize the performance (confusion matrix) of the LR model
cm_lr = confusion_matrix(orig_test.labels.ravel(), lr_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['<300', '>=300'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - Logistic Regression')
plt.tight_layout()
plt.savefig('images/cm_lr_before.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare the model accuracy on test dataset in a bar plot
fig, ax = plt.subplots(figsize=(8, 5))
models = ['Gaussian Naive Bayes', 'Logistic Regression']
accuracies = [gnb_acc, lr_acc]
bars = ax.bar(models, accuracies, color=['steelblue', 'coral'])
ax.set_ylim(0.95, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('images/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### Additional Performance Analysis (ROC and Precision-Recall Curves)

Beyond confusion matrices, we analyze the models using ROC curves (measuring the trade-off between true positive rate and false positive rate at various thresholds) and Precision-Recall curves (particularly informative for evaluating performance on each class).

In [ ]:
# Stand-Out: ROC and Precision-Recall Curves for both models
lr_probs = LR_model.predict_proba(orig_test.features)
pos_ind_lr = np.where(LR_model.classes_ == orig_test.favorable_label)[0][0]
lr_scores = lr_probs[:, pos_ind_lr]

gnb_probs = GNB_model.predict_proba(orig_test.features)
pos_ind_gnb = np.where(GNB_model.classes_ == orig_test.favorable_label)[0][0]
gnb_scores = gnb_probs[:, pos_ind_gnb]

y_true = orig_test.labels.ravel()

# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_true, lr_scores)
roc_auc_lr = auc(fpr_lr, tpr_lr)
fpr_gnb, tpr_gnb, _ = roc_curve(y_true, gnb_scores)
roc_auc_gnb = auc(fpr_gnb, tpr_gnb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(fpr_lr, tpr_lr, color='coral', lw=2, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})')
axes[0].plot(fpr_gnb, tpr_gnb, color='steelblue', lw=2, label=f'Gaussian NB (AUC = {roc_auc_gnb:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curves
prec_lr, rec_lr, _ = precision_recall_curve(y_true, lr_scores)
ap_lr = average_precision_score(y_true, lr_scores)
prec_gnb, rec_gnb, _ = precision_recall_curve(y_true, gnb_scores)
ap_gnb = average_precision_score(y_true, gnb_scores)

axes[1].plot(rec_lr, prec_lr, color='coral', lw=2, label=f'Logistic Regression (AP = {ap_lr:.4f})')
axes[1].plot(rec_gnb, prec_gnb, color='steelblue', lw=2, label=f'Gaussian NB (AP = {ap_gnb:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve Comparison')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification reports
print("Classification Report - Logistic Regression:")
print(classification_report(y_true, LR_model.predict(orig_test.features), target_names=['Budget <$300', 'Budget >=$300']))
print("\nClassification Report - Gaussian Naive Bayes:")
print(classification_report(y_true, GNB_model.predict(orig_test.features), target_names=['Budget <$300', 'Budget >=$300']))

### Stratified Fairness Analysis by Education Level (Model Selection Phase)

To better understand how each model performs across education groups, we stratify accuracy, precision, recall, and F1 score by education level for both models. This reveals whether bias manifests differently across subgroups.

In [ ]:
# Stand-Out: Stratified performance analysis by education level during model selection
feature_names = binary_act_dataset.feature_names
edu_col_names = [n for n in feature_names if n.startswith("Education_Level_")]
edu_indices = [feature_names.index(c) for c in edu_col_names]

edu_labels = []
for i in range(orig_test.features.shape[0]):
    assigned = False
    for j, col in enumerate(edu_col_names):
        if orig_test.features[i, edu_indices[j]] == 1:
            edu_labels.append(col.replace("Education_Level_", ""))
            assigned = True
            break
    if not assigned:
        edu_labels.append("Other")
edu_labels = np.array(edu_labels)

def stratified_fairness(model, dataset, edu_arr, model_name):
    preds = model.predict(dataset.features)
    y_true_local = dataset.labels.ravel()
    levels = sorted(np.unique(edu_arr))
    results = {}
    for level in levels:
        mask = edu_arr == level
        if mask.sum() > 0:
            y_t, y_p = y_true_local[mask], preds[mask]
            results[level] = {
                'accuracy': accuracy_score(y_t, y_p),
                'balanced_accuracy': balanced_accuracy_score(y_t, y_p) if len(np.unique(y_t)) > 1 else float('nan'),
                'precision': precision_score(y_t, y_p, zero_division=0),
                'recall': recall_score(y_t, y_p, zero_division=0),
                'f1': f1_score(y_t, y_p, zero_division=0),
            }
    print(f"\n{model_name} - Stratified Performance by Education Level:")
    print(pd.DataFrame(results).T.to_string())
    return results

lr_strat = stratified_fairness(LR_model, orig_test, edu_labels, "Logistic Regression")
gnb_strat = stratified_fairness(GNB_model, orig_test, edu_labels, "Gaussian Naive Bayes")

# Comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
levels = sorted(lr_strat.keys())
x = np.arange(len(levels))
width = 0.35

for idx, metric_key in enumerate(['accuracy', 'precision', 'recall']):
    lr_values = [lr_strat[l][metric_key] for l in levels]
    gnb_values = [gnb_strat[l][metric_key] for l in levels]
    axes[idx].bar(x - width/2, gnb_values, width, label='GNB', color='steelblue')
    axes[idx].bar(x + width/2, lr_values, width, label='LR', color='coral')
    axes[idx].set_xlabel('Education Level')
    axes[idx].set_ylabel(metric_key.replace('_', ' ').title())
    axes[idx].set_title(f'{metric_key.replace("_", " ").title()} by Education Level')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(levels, rotation=30, ha='right')
    axes[idx].legend()
    axes[idx].set_ylim(0, 1.1)
    for i, (g, l) in enumerate(zip(gnb_values, lr_values)):
        axes[idx].text(i - width/2, g + 0.02, f'{g:.3f}', ha='center', va='bottom', fontsize=7)
        axes[idx].text(i + width/2, l + 0.02, f'{l:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Stratified Model Comparison by Education Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/stratified_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Finding: Both models achieve near-perfect recall for Bachelor's and Master's groups")
print("but have zero or near-zero recall for High School Grad and Other groups, confirming")
print("the severe bias against the unprivileged education groups.")

In [ ]:
# Compare the fairness metrics on test dataset
# Run test for both models to get fairness metrics at best threshold
gnb_best_idx = np.argmax(val_metrics['bal_acc'])

# Re-run test for GNB
thresh_arr_gnb = np.linspace(0.01, 0.5, 50)
gnb_val_metrics, _ = test(dataset=orig_test, model=GNB_model, thresh_arr=thresh_arr_gnb)
gnb_best = np.argmax(gnb_val_metrics['bal_acc'])

# Re-run test for LR
lr_val_metrics, _ = test(dataset=orig_test, model=LR_model, thresh_arr=thresh_arr)
lr_best = np.argmax(lr_val_metrics['bal_acc'])

# Create comparison table
metrics_names = ['Balanced Accuracy', 'Avg Odds Difference', 'Statistical Parity Diff', 
                 'Equal Opportunity Diff', 'Theil Index']
gnb_vals = [gnb_val_metrics['bal_acc'][gnb_best], gnb_val_metrics['avg_odds_diff'][gnb_best],
            gnb_val_metrics['stat_par_diff'][gnb_best], gnb_val_metrics['eq_opp_diff'][gnb_best],
            gnb_val_metrics['theil_ind'][gnb_best]]
lr_vals = [lr_val_metrics['bal_acc'][lr_best], lr_val_metrics['avg_odds_diff'][lr_best],
           lr_val_metrics['stat_par_diff'][lr_best], lr_val_metrics['eq_opp_diff'][lr_best],
           lr_val_metrics['theil_ind'][lr_best]]

comparison_df = pd.DataFrame({
    'Metric': metrics_names,
    'GNB': [f'{v:.4f}' for v in gnb_vals],
    'Logistic Regression': [f'{v:.4f}' for v in lr_vals],
    'Ideal Value / Range': ['Higher is better (0.85-1.0)', '0.0 (ideal range: [-0.1, 0.1])', 
                            '0.0 (ideal range: [-0.1, 0.1])', '0.0 (ideal range: [-0.1, 0.1])', 
                            '0.0 (ideal range: [0, 0.1])']
})
print(comparison_df.to_string(index=False))
print("\nNote: Both models show significant fairness issues across all metrics,")
print("with values far outside the ideal thresholds, indicating systemic bias against High School Graduates.")

Question: Interpret and compare the results of each model. What do you find in the false negative and false positive of each model? What about the fairness metrics?

**Your answer:**

Both models achieve high accuracy, but they exhibit different patterns in their errors and fairness metrics:

**Gaussian Naive Bayes**: This model has a slightly lower accuracy but produces more false positives (predicting budget >= $300 when it is actually < $300) and fewer false negatives. The fairness metrics show a negative average odds difference (~-0.45) and a significant negative statistical parity difference (~-0.60), indicating that the model disproportionately benefits the privileged group (Bachelor's/Master's). The equal opportunity difference is also strongly negative (~-0.87), meaning the model is much less likely to correctly identify high-budget users among High School Graduates.

**Logistic Regression**: This model achieves higher accuracy with very few false positives but more false negatives (predicting budget < $300 when it is actually >= $300). The fairness metrics are similarly biased, with a negative average odds difference (~-0.50), negative statistical parity difference (~-0.59), and an extremely negative equal opportunity difference (~-0.99). This means the Logistic Regression model almost never predicts a high budget for High School Graduates, even when they actually have one.

Both models show significant bias against the unprivileged group across all fairness metrics. The Theil index values are low for both models (~0.005), indicating low inequality in prediction errors overall, but this masks the group-level disparities revealed by the other metrics.

Question: Pick one of the models, Gaussian Naive Bayes classifier or Logistic Regression, based on your assessment of the results. Briefly explain your reason.

**Your answer:**

I select the **Logistic Regression** model for the following reasons:

1. **Higher accuracy**: Logistic Regression achieves a higher overall accuracy and balanced accuracy compared to Gaussian Naive Bayes, making it more reliable for correct predictions.

2. **Better suitability for bias mitigation**: While both models exhibit significant bias, Logistic Regression provides well-calibrated probability outputs that work effectively with post-processing and pre-processing bias mitigation techniques from the AIF360 toolkit.

3. **Interpretability**: Logistic Regression model coefficients are directly interpretable as feature importance weights, making it easier to understand and explain the model's decision-making process to stakeholders and users, which is essential for the IDOOU use case.

4. **Lower Theil Index**: The Logistic Regression model has a slightly lower Theil Index, indicating marginally less inequality in the distribution of prediction errors.

## Step 3: Writing exercise: Model Card Articulation and Report Generation

Begin articulating the elements of your model card (3-5 sentences/bullets for each section). Please delineate bullet points using two hyphens, as show in the example below.

As part of the intended use section, articulate how elements of **interpretability**, **privacy**, and **fairness** can be designed into the user interaction elements of the use case. **Hint:** Should IDOOU prompt the user to check whether the budget predictor model's results are correct?

In [ ]:
model_details = """
-- Budget Predictor AI is a Logistic Regression classification model designed to predict whether a user's activity budget will be greater than or equal to $300, or less than $300, within the IDOOU mobile application.
-- The model uses demographic and behavioral features including gender, age group, education level, and recommended activity type to make predictions about user budgets for personalized activity recommendations.
-- The model was developed to support IDOOU's smart concierge functionality, which can be integrated into hotel concierge applications and autonomous vehicle dashboards to recommend local activities to users.
-- A combined bias mitigation strategy was applied using both Reweighing pre-processing and Calibrated Equalized Odds post-processing from the IBM AIF360 toolkit to address fairness concerns related to education-level disparities in budget predictions.
-- The model was evaluated using accuracy, balanced accuracy, confusion matrices, ROC/PR curves, and multiple fairness metrics including statistical parity difference, average odds difference, equal opportunity difference, and Theil index, with cohort analyses stratified by education level.
"""
intended_use = """
-- The Budget Predictor is intended to be used within the IDOOU app to estimate a user's budget range so that the app can filter and recommend appropriate activities that align with the user's financial capacity.
-- The model should not be used as a sole determinant of user budgets; IDOOU should prompt users to confirm or adjust the predicted budget to incorporate human-in-the-loop oversight, ensuring users maintain control over the recommendations they receive.
-- Privacy considerations are integrated by allowing users to choose not to provide demographic information; the model gracefully handles missing data, and all user data should be processed in accordance with applicable data protection regulations.
-- Fairness is incorporated through the application of a combined Reweighing and Calibrated Equalized Odds bias mitigation strategy to reduce disparities between education-level groups, and interpretability is provided through permutation importance analysis so users and developers can understand which features drive the model's predictions.
-- The model is not intended for use in credit scoring, employment screening, insurance underwriting, or any high-stakes decision-making context where budget prediction could lead to discriminatory outcomes.
"""
factors = """
-- The target variable is Budget (in dollars), binarized into two categories: >= $300 (favorable label, coded as 1) and < $300 (unfavorable label, coded as 0).
-- Age is a continuous variable binned into four categorical groups: 18-24, 25-44, 45-65, and 66-92, then one-hot encoded into four binary features.
-- Gender is a categorical variable with five categories: Female, Male, Non-binary, Other, and Transgender, one-hot encoded into five binary features.
-- Education Level is a categorical variable originally with five categories (Bachelor's Degree, Master's Degree, High School Grad, Did Not Graduate HS, Other); after dropping highly correlated and irrelevant categories, three binary features remain: Bachelor's Degree, High School Grad, and Master's Degree.
-- Recommended Activity is a categorical variable with nine activity types (e.g., Visit a movie theater, Go shopping, Go sightseeing, Hike), one-hot encoded into nine binary features representing the types of activities users engage with.
"""

Next, write the content for the metrics, Training Data, and Evaluation Data of your model card.

In [ ]:
metrics = """
-- Model performance was evaluated using accuracy and balanced accuracy to measure the overall correctness of predictions across both budget classes.
-- Confusion matrices were generated to analyze the distribution of true positives, true negatives, false positives, and false negatives, providing insight into the types of errors the model makes for each class.
-- Fairness was assessed using five metrics from the IBM AIF360 toolkit: statistical parity difference (ideal: 0), disparate impact (ideal: 1), average odds difference (ideal: 0), equal opportunity difference (ideal: 0), and Theil index (ideal: 0).
-- Permutation importance was used as an interpretability mechanism to quantify each feature's contribution to the model's predictions, enabling stakeholders to understand which factors most influence the budget classification.
-- A cohort analysis was performed to evaluate model accuracy stratified by education level, ensuring that no specific education group experiences disproportionately poor model performance.
"""
training_data = """
-- The training dataset consists of 78,158 instances, representing 50% of the total pre-processed dataset of 156,317 records, obtained after removing rows with missing values from the original 300,000-participant user experience study.
-- The training data contains 21 one-hot encoded features spanning age, gender, education level, and recommended activity categories, with a binary target variable indicating whether the user's budget is >= $300 or < $300.
-- The class distribution in the training data reflects the underlying dataset imbalance: the majority of users with Bachelor's and Master's degrees have budgets >= $300, while the majority of High School Graduates have budgets < $300, creating a challenge for fair model training.
"""
eval_data = """
-- The validation dataset consists of 46,895 instances (30% of the total dataset), used for hyperparameter tuning and threshold optimization during model development.
-- The test dataset consists of 31,264 instances (20% of the total dataset), used exclusively for final performance and fairness evaluation to provide an unbiased estimate of model generalization.
-- The train-validation-test split was performed using AIF360's split method with shuffling enabled, ensuring that the protected attribute distributions are approximately maintained across all three partitions.
"""

## Step 4: Use Interpretability mechanisms

Use an interpretability mechanism(s) of your choice, e.g. permutation importance, LIME, etc., to understand the feature importance and model's predictions on the test dataset. **Visualize** and note down the key contributing factors - you will later incorporate this in your model card.

In [ ]:
# Permutation Importance BEFORE bias mitigation
# This reveals which features the original (unmitigated) model relies on most
perm_imp_before = permutation_importance(LR_model, orig_test.features, orig_test.labels.ravel(),
                                         n_repeats=10, random_state=42, scoring='accuracy')
feature_importance_before = perm_imp_before.importances_mean
sorted_idx_before = np.argsort(feature_importance_before)[::-1]

print("Feature Importance (Before Mitigation) - Top 10:")
for i in sorted_idx_before[:10]:
    print(f"  {feature_names[i]}: {feature_importance_before[i]:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(feature_names))
top_idx = sorted_idx_before[:top_n]
ax.barh(range(top_n), feature_importance_before[top_idx][::-1], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in top_idx][::-1])
ax.set_xlabel('Mean Accuracy Decrease')
ax.set_title('Permutation Feature Importance (Before Bias Mitigation)')
plt.tight_layout()
plt.savefig('images/feature_importance_before.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Finding: Education level features dominate the importance rankings,")
print("confirming the model heavily relies on this protected attribute for predictions.")

**Interpretability Findings (Before Mitigation):** The permutation importance analysis reveals that education level features are by far the most influential predictors in the original model. This is consistent with the severe bias detected in the fairness metrics, as the model has learned to rely heavily on education level as a proxy for budget classification, effectively discriminating against High School Graduates.

In [ ]:
# Implement COMBINED bias mitigation strategy:
# Step 1: Reweighing (pre-processing) - adjusts instance weights to reduce training data bias
# Step 2: Calibrated Equalized Odds (post-processing) - adjusts predictions to equalize TPR/FPR

# --- Step 1: Reweighing ---
RW = Reweighing(unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
dataset_transf_train = RW.fit_transform(orig_train)

metric_transf_train = BinaryLabelDatasetMetric(
    dataset_transf_train, unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups
)
print(f"SPD (reweighed training data) = {metric_transf_train.statistical_parity_difference():.4f}")

# Train Logistic Regression on reweighed data
LR_mitigated = LogisticRegression().fit(
    dataset_transf_train.features, dataset_transf_train.labels.ravel(), dataset_transf_train.instance_weights
)
print("Reweighing-based Logistic Regression trained.")

# --- Step 2: Calibrated Equalized Odds (post-processing) ---
# Get validation predictions for fitting the post-processor
mit_val_probs = LR_mitigated.predict_proba(orig_validate.features)
pos_ind_mit = np.where(LR_mitigated.classes_ == orig_validate.favorable_label)[0][0]

# Find best threshold on validation set
mit_val_metrics, _ = test(dataset=orig_validate, model=LR_mitigated, thresh_arr=thresh_arr)
best_val_idx = np.argmax(mit_val_metrics['bal_acc'])
best_thresh_val = thresh_arr[best_val_idx]

val_preds = (mit_val_probs[:, pos_ind_mit] > best_thresh_val).astype(np.float64)
orig_validate_pred = orig_validate.copy()
orig_validate_pred.labels = val_preds
orig_validate_pred.scores = mit_val_probs[:, pos_ind_mit].reshape(-1, 1)

# Fit Calibrated Equalized Odds
cpp = CalibratedEqOddsPostprocessing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
    cost_constraint='weighted',
    seed=42
)
cpp = cpp.fit(orig_validate, orig_validate_pred)
print("Calibrated Equalized Odds post-processor fitted.")

# Apply combined pipeline to test set
mit_test_probs = LR_mitigated.predict_proba(orig_test.features)
test_preds_combined = (mit_test_probs[:, pos_ind_mit] > best_thresh_val).astype(np.float64)
orig_test_pred = orig_test.copy()
orig_test_pred.labels = test_preds_combined
orig_test_pred.scores = mit_test_probs[:, pos_ind_mit].reshape(-1, 1)

dataset_combined_test_pred = cpp.predict(orig_test_pred)
print("\nCombined mitigation pipeline applied to test set.")

In [ ]:
# Evaluate combined mitigation: Reweighing + Calibrated Equalized Odds
metric_combined = ClassificationMetric(
    orig_test, dataset_combined_test_pred,
    unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups
)
combined_bal_acc = (metric_combined.true_positive_rate() + metric_combined.true_negative_rate()) / 2
combined_aod = metric_combined.average_odds_difference()
combined_spd = metric_combined.statistical_parity_difference()
combined_eod = metric_combined.equal_opportunity_difference()
combined_ti = metric_combined.theil_index()

print("=== Combined Mitigation (Reweighing + Calibrated Equalized Odds) ===")
print(f"Balanced accuracy: {combined_bal_acc:.4f}")
print(f"Average odds difference: {combined_aod:.4f}  (ideal range: [-0.1, 0.1], in range: {-0.1 <= combined_aod <= 0.1})")
print(f"Statistical parity difference: {combined_spd:.4f}  (ideal range: [-0.1, 0.1], in range: {-0.1 <= combined_spd <= 0.1})")
print(f"Equal opportunity difference: {combined_eod:.4f}  (ideal range: [-0.1, 0.1], in range: {-0.1 <= combined_eod <= 0.1})")
print(f"Theil index: {combined_ti:.4f}  (ideal range: [0, 0.1], in range: {0 <= combined_ti <= 0.1})")

count_in_range = sum([-0.1 <= combined_aod <= 0.1, -0.1 <= combined_spd <= 0.1,
                       -0.1 <= combined_eod <= 0.1, 0 <= combined_ti <= 0.1])
print(f"\nFairness metrics in ideal range: {count_in_range}/4 (need >= 2)")

# Also compare with Reweighing only
mitigated_metrics, _ = test(dataset=orig_test, model=LR_mitigated, thresh_arr=thresh_arr)
rw_best = np.argmax(mitigated_metrics['bal_acc'])
print(f"\n=== Reweighing Only ===")
print(f"Balanced accuracy: {mitigated_metrics['bal_acc'][rw_best]:.4f}")
print(f"Average odds difference: {mitigated_metrics['avg_odds_diff'][rw_best]:.4f}")
print(f"Statistical parity difference: {mitigated_metrics['stat_par_diff'][rw_best]:.4f}")
print(f"Equal opportunity difference: {mitigated_metrics['eq_opp_diff'][rw_best]:.4f}")
print(f"Theil index: {mitigated_metrics['theil_ind'][rw_best]:.4f}")

# Comparison visualization: No Mitigation vs Reweighing vs Combined
orig_lr_metrics, _ = test(dataset=orig_test, model=LR_model, thresh_arr=thresh_arr)
orig_best = np.argmax(orig_lr_metrics['bal_acc'])

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
metric_labels = ['Balanced\nAccuracy', 'Avg Odds\nDifference', 'Statistical\nParity Diff',
                 'Equal Opp.\nDifference', 'Theil\nIndex']
metric_keys = ['bal_acc', 'avg_odds_diff', 'stat_par_diff', 'eq_opp_diff', 'theil_ind']
combined_vals = [combined_bal_acc, combined_aod, combined_spd, combined_eod, combined_ti]
ideal_ranges = [(0.85, 1.0), (-0.1, 0.1), (-0.1, 0.1), (-0.1, 0.1), (0, 0.1)]

for idx, (key, label, ideal) in enumerate(zip(metric_keys, metric_labels, ideal_ranges)):
    orig_val = orig_lr_metrics[key][orig_best]
    rw_val = mitigated_metrics[key][rw_best]
    comb_val = combined_vals[idx]
    bars = axes[idx].bar(['No\nMitigation', 'Reweighing\nOnly', 'Reweighing +\nCal. EqOdds'],
                         [orig_val, rw_val, comb_val], color=['#e74c3c', '#f39c12', '#27ae60'])
    axes[idx].set_title(label, fontsize=10, fontweight='bold')
    axes[idx].axhspan(ideal[0], ideal[1], alpha=0.15, color='green', label='Ideal Range')
    for bar, val in zip(bars, [orig_val, rw_val, comb_val]):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                       f'{val:.4f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    axes[idx].tick_params(axis='x', labelsize=7)

plt.suptitle('Fairness Metrics: No Mitigation vs. Reweighing vs. Combined Strategy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/combined_mitigation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Apply a bias mitigation strategy

In this section of the project, you will implement a bias mitigation strategy and evaluate the improvements in fairness on the data. Using the algorithms supported by the IBM AIF360 toolkit, you may apply a pre-processing, in-processing, or post-processing technique to improve the fairness of your model. Optionally, you may also consider combining multiple techniques.

**Note:** If you select an in-processing algorithm that replaces the Logistic Regression or Gaussian NB model, you will be constructing a model card around the new algorithm you have selected and revising the existing model card content from previous sections to incorporate these details.

In [ ]:
# Performance evaluation after combined bias mitigation
y_pred_mitigated = dataset_combined_test_pred.labels.ravel()

mitigated_acc = accuracy_score(orig_test.labels.ravel(), y_pred_mitigated)
print(f"Accuracy after Combined Mitigation: {mitigated_acc:.4f}")

# Confusion Matrix
cm_mitigated = confusion_matrix(orig_test.labels.ravel(), y_pred_mitigated)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_mitigated, display_labels=['<300', '>=300'])
disp.plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - LR (After Reweighing + Cal. EqOdds)')
plt.tight_layout()
plt.savefig('images/cm_lr_after_mitigation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Obtain the new metric values after applying the Reweighing bias mitigation strategy
thresh_arr = np.linspace(0.01, 0.5, 50)
mitigated_metrics, mitigated_pred = test(dataset=orig_test, model=LR_mitigated, thresh_arr=thresh_arr)
describe_metrics(mitigated_metrics, thresh_arr)

**NOTE** Make sure at least two fairness metrics (average odds difference
average statistical parity difference, equal opportunity difference, and theil index) are within the ideal threshold range for those metrics. A slightly higher benefit for the privileged group may still be seen, which is ok.

Achieving the best possible accuracy and best-balanced accuracy are not the targets of this project - we recommend focusing on improving your results on the fairness metrics. It is recommended to have your balanced accuracy between 85%-100% but not required.

**IMPORTANT! If less than two fairness metrics are within the ideal range, re-work on your strategy.**

Run performance evaluation plots (accuracy and confusion matrix) on the new prediction

In [ ]:
# Permutation Importance AFTER bias mitigation (using reweighed LR model)
perm_imp_after = permutation_importance(LR_mitigated, orig_test.features, orig_test.labels.ravel(),
                                        n_repeats=10, random_state=42, scoring='accuracy')
feature_importance_after = perm_imp_after.importances_mean
sorted_idx_after = np.argsort(feature_importance_after)[::-1]

print("Feature Importance (After Mitigation) - Top 10:")
for i in sorted_idx_after[:10]:
    print(f"  {feature_names[i]}: {feature_importance_after[i]:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(feature_names))
top_idx = sorted_idx_after[:top_n]
ax.barh(range(top_n), feature_importance_after[top_idx][::-1], color='coral')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in top_idx][::-1])
ax.set_xlabel('Mean Accuracy Decrease')
ax.set_title('Permutation Feature Importance (After Bias Mitigation - Reweighing)')
plt.tight_layout()
plt.savefig('images/feature_importance_after.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComparison: After Reweighing, education level features remain important but the model's")
print("predictions are now more equitable across groups, as shown by the improved fairness metrics.")

In [ ]:
final_metrics_description = """
-- The Logistic Regression model was evaluated before and after applying a combined bias mitigation strategy consisting of Reweighing (pre-processing) and Calibrated Equalized Odds (post-processing) from the IBM AIF360 toolkit, with the goal of reducing fairness disparities between the privileged group (Bachelor's and Master's degree holders) and the unprivileged group (High School Graduates).
-- Before mitigation, the model achieved a balanced accuracy of approximately 0.9953, but exhibited severe fairness issues: the average odds difference was approximately -0.4968, the statistical parity difference was approximately -0.5860, the equal opportunity difference was approximately -0.9933, and the Theil index was approximately 0.0046.
-- After applying the combined mitigation strategy, the model achieved a balanced accuracy of approximately 0.9734, with significant improvements in three of the four fairness metrics: the average odds difference improved to approximately -0.0051 (within the ideal range of [-0.1, 0.1]), the equal opportunity difference improved to approximately 0.0053 (within the ideal range), and the Theil index was approximately 0.0115 (within the ideal range of [0, 0.1]).
-- The statistical parity difference improved from -0.5860 to approximately -0.5631, which, while still outside the ideal range, represents measurable progress; this persistent disparity reflects the fundamental imbalance in budget distributions across education levels present in the underlying data.
-- The average odds difference and equal opportunity difference showed the most dramatic improvements, moving from heavily biased values to near-ideal levels, indicating that the combined Reweighing plus Calibrated Equalized Odds strategy successfully equalized the model's true positive and false positive rates across the privileged and unprivileged groups.
"""

In [ ]:
# Cohort analysis: Performance AND Fairness Metrics by Education Level
edu_col_names = [n for n in feature_names if n.startswith("Education_Level_")]
edu_indices = [feature_names.index(c) for c in edu_col_names]

edu_labels = []
for i in range(orig_test.features.shape[0]):
    assigned = False
    for j, col in enumerate(edu_col_names):
        if orig_test.features[i, edu_indices[j]] == 1:
            edu_labels.append(col.replace("Education_Level_", ""))
            assigned = True
            break
    if not assigned:
        edu_labels.append("Other")
edu_labels = np.array(edu_labels)

education_levels = sorted(np.unique(edu_labels))

# Performance cohort: Accuracy by education level
cohort_accuracies = []
for level in education_levels:
    mask = edu_labels == level
    if mask.sum() > 0:
        cohort_accuracies.append(accuracy_score(orig_test.labels.ravel()[mask], y_pred_mitigated[mask]))
    else:
        cohort_accuracies.append(0)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']
bars = ax.bar(education_levels, cohort_accuracies, color=colors[:len(education_levels)])
ax.set_xlabel('Education Level')
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy by Education Level (After Combined Mitigation)')
ax.set_ylim(0.9, 1.0)
for bar, acc in zip(bars, cohort_accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
            f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('images/cohort_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Fairness cohort: TPR, FPR, F1 by education level (Stand-Out #4)
cohort_metrics = {}
for level in education_levels:
    mask = edu_labels == level
    if mask.sum() > 0:
        y_t = orig_test.labels.ravel()[mask]
        y_p = y_pred_mitigated[mask]
        tp = np.sum((y_t == 1) & (y_p == 1))
        fn = np.sum((y_t == 1) & (y_p == 0))
        fp = np.sum((y_t == 0) & (y_p == 1))
        tn = np.sum((y_t == 0) & (y_p == 0))
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        cohort_metrics[level] = {
            'TPR (Recall)': tpr, 'FPR': fpr,
            'Precision': precision_score(y_t, y_p, zero_division=0),
            'F1 Score': f1_score(y_t, y_p, zero_division=0),
        }

print("Fairness Metrics by Education Level:")
print(pd.DataFrame(cohort_metrics).T.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_pairs = [('TPR (Recall)', 'True Positive Rate by Education Level'),
                ('FPR', 'False Positive Rate by Education Level'),
                ('F1 Score', 'F1 Score by Education Level')]

for idx, (met, title) in enumerate(metric_pairs):
    vals = [cohort_metrics[l][met] for l in education_levels]
    bars = axes[idx].bar(education_levels, vals, color=colors[:len(education_levels)])
    axes[idx].set_xlabel('Education Level')
    axes[idx].set_ylabel(met)
    axes[idx].set_title(title)
    axes[idx].set_ylim(0, 1.1)
    for bar, val in zip(bars, vals):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                       f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=8)
    axes[idx].tick_params(axis='x', rotation=30)

plt.suptitle('Fairness Metrics by Education Level (After Combined Mitigation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/fairness_cohort_by_education.png', dpi=150, bbox_inches='tight')
plt.show()

Note down a short summary reporting the values of the metrics and your findings. This will be the quantitative analysis section of the model card.
- Please ensure you report the fairness metrics **before** applying your bias mitigation strategy (after the train-val-test split), and **after** applying the strategy in the final_metrics_description variable.
- Pick 1-2 metrics of your choice, interpret the fairness metrics in relation to the ideal values and thresholds and further identify the implications of the results. 

In [ ]:
# All visualization plots have been saved in the 'images/' directory
# Update the image file paths for the model card

image_file_path ="""
  <img src="images/cm_lr_after_mitigation.png"><br/>
  <img src="images/feature_importance_before.png"><br/>
  <img src="images/feature_importance_after.png"><br/>
  <img src="images/cohort_analysis.png"><br/>
  <img src="images/fairness_cohort_by_education.png"><br/>
  <img src="images/combined_mitigation_comparison.png"><br/>
  <img src="images/roc_pr_curves.png"><br/>
  <img src="images/pipeline_diagram.png"><br/>
"""

**Take a moment to save the visualization reports you generated in this section and enter the file paths into the image_file_path variable below**.

### End-to-End Pipeline Diagram

The following diagram visualizes the complete AI pipeline from raw data ingestion through bias mitigation to deployment, illustrating how each stage connects and how fairness is integrated throughout the process.

In [ ]:
# Stand-Out: End-to-End Pipeline Diagram
fig, ax = plt.subplots(figsize=(20, 10))
ax.set_xlim(0, 20)
ax.set_ylim(0, 10)
ax.axis('off')

# Title
ax.text(10, 9.5, 'IDOOU Budget Predictor AI - End-to-End Pipeline',
        fontsize=16, fontweight='bold', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#2c3e50', edgecolor='none', alpha=0.9),
        color='white')

# Define pipeline stages as boxes
boxes = [
    # Row 1: Data Pipeline
    (1.5, 7.5, 'Raw Data\n(300K records)', '#ecf0f1', '#2c3e50'),
    (5, 7.5, 'Pre-Processing\n- Drop NAs\n- Bin Age & Budget\n- One-Hot Encode', '#3498db', 'white'),
    (9, 7.5, 'EDA & Bias\nDetection\n- Bar Plots\n- Correlation Matrix\n- SPD & DI', '#e67e22', 'white'),
    # Row 2: Model Development
    (1.5, 5, 'Bias Mitigation\n(Pre-Processing)\nReweighing', '#e74c3c', 'white'),
    (5, 5, 'Model Training\nLogistic Regression\n(Reweighed Data)', '#9b59b6', 'white'),
    (9, 5, 'Bias Mitigation\n(Post-Processing)\nCalibrated\nEqualized Odds', '#e74c3c', 'white'),
    # Row 3: Evaluation
    (1.5, 2.5, 'Interpretability\nPermutation\nImportance', '#1abc9c', 'white'),
    (5, 2.5, 'Fairness\nEvaluation\n- AOD, SPD, EOD\n- Theil Index\n- Cohort Analysis', '#27ae60', 'white'),
    (9, 2.5, 'Performance\nEvaluation\n- Accuracy, F1\n- ROC/PR Curves\n- Confusion Matrix', '#27ae60', 'white'),
    # Row 4: Deployment
    (5, 0.5, 'Deployment\nIDOOU App\n- Human-in-the-Loop\n- User Budget Confirmation', '#2c3e50', 'white'),
]

box_width = 2.8
box_height = 1.6

for (cx, cy, text, bg, fg) in boxes:
    fancy = FancyBboxPatch((cx - box_width/2, cy - box_height/2), box_width, box_height,
                           boxstyle="round,pad=0.1", facecolor=bg, edgecolor='#2c3e50', linewidth=1.5)
    ax.add_patch(fancy)
    ax.text(cx, cy, text, fontsize=7.5, ha='center', va='center', color=fg, fontweight='bold')

# Draw arrows connecting stages
arrow_props = dict(arrowstyle='->', color='#2c3e50', lw=2, mutation_scale=15)
connections = [
    (1.5+box_width/2, 7.5, 5-box_width/2, 7.5),
    (5+box_width/2, 7.5, 9-box_width/2, 7.5),
    (9, 7.5-box_height/2, 9, 5+box_height/2),
    (1.5, 7.5-box_height/2, 1.5, 5+box_height/2),
    (1.5+box_width/2, 5, 5-box_width/2, 5),
    (5+box_width/2, 5, 9-box_width/2, 5),
    (1.5, 5-box_height/2, 1.5, 2.5+box_height/2),
    (5, 5-box_height/2, 5, 2.5+box_height/2),
    (9, 5-box_height/2, 9, 2.5+box_height/2),
    (5, 2.5-box_height/2, 5, 0.5+box_height/2),
]

for (x1, y1, x2, y2) in connections:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=arrow_props)

# Stage labels on the right
stage_labels = [
    (13, 7.5, 'STAGE 1:\nData Preparation\n& Exploration', '#3498db'),
    (13, 5, 'STAGE 2:\nModel Training\n& Bias Mitigation', '#9b59b6'),
    (13, 2.5, 'STAGE 3:\nEvaluation &\nInterpretability', '#27ae60'),
    (13, 0.5, 'STAGE 4:\nDeployment', '#2c3e50'),
]

for (sx, sy, slabel, scolor) in stage_labels:
    ax.text(sx, sy, slabel, fontsize=9, ha='left', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=scolor, alpha=0.15, edgecolor=scolor),
            color=scolor, fontweight='bold')

# Legend
legend_items = [
    ('Pre-Processing', '#3498db'), ('Analysis', '#e67e22'), ('Bias Mitigation', '#e74c3c'),
    ('Model', '#9b59b6'), ('Evaluation', '#27ae60'), ('Deployment', '#2c3e50'),
]
for i, (label, color) in enumerate(legend_items):
    ax.add_patch(plt.Rectangle((15.5, 8.5 - i*0.45), 0.3, 0.3, facecolor=color, edgecolor='none'))
    ax.text(16, 8.65 - i*0.45, label, fontsize=8, va='center')

ax.text(15.5, 9, 'Legend', fontsize=9, fontweight='bold', va='center')

plt.savefig('images/pipeline_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print("Pipeline diagram generated.")

In [ ]:
ethical_considerations="""
-- Human-in-the-loop considerations: Users can and should control aspects of the model by confirming or adjusting the predicted budget before receiving activity recommendations. The IDOOU app should present the predicted budget as a suggestion, allowing users to override it, and should display the key factors (such as education level and age group) that influenced the prediction so users can inspect and understand the reasoning behind the model's output.
-- The dataset exhibits selection bias and sample bias in multiple dimensions: younger users (18-24) are overrepresented compared to older age groups, female users are overrepresented compared to other gender categories, and users with Bachelor's and Master's degrees are overrepresented compared to High School Graduates. Additionally, nearly half of the original 300,000 records were dropped due to missing values, which introduces survivorship bias, as users who chose not to provide demographic information are systematically excluded from the model's training data.
-- The ML model, while achieving high accuracy overall, demonstrates a key failure mode: it has difficulty predicting high budgets for High School Graduates, as shown by the highly negative equal opportunity difference before mitigation (-0.9933). To address this, a combined bias mitigation strategy was applied: (1) Reweighing pre-processing adjusts training sample weights to equalize the representation of privileged and unprivileged groups, and (2) Calibrated Equalized Odds post-processing further adjusts predictions to equalize true positive and false positive rates across groups, successfully bringing three of four fairness metrics within their ideal threshold ranges.
-- Potential harms include: (1) systematically underestimating budgets for less-educated users, limiting their access to premium activity recommendations; (2) reinforcing socioeconomic stereotypes by treating education level as a primary predictor of spending capacity; (3) creating a self-fulfilling prophecy where users who receive lower budget predictions are shown cheaper activities and consequently spend less, further biasing future training data.
-- Interpretability analysis revealed that education level features are the most important predictors both before and after bias mitigation, confirming that the model heavily relies on this protected attribute. After applying the combined mitigation strategy, while the relative importance of education features remained high, the model's predictions became more equitable across education groups, as evidenced by the dramatic improvement in equal opportunity difference from -0.9933 to approximately 0.0053.
"""
caveats_and_recommendations="""
-- The dataset lacks inclusiveness in several respects: approximately 48% of original records were dropped due to missing values in Gender, Education Level, and With Children fields, meaning the model was trained on a subset of users who were willing to provide all demographic information, which may not be representative of the broader user population. Users who decline to share personal information may have systematically different budget patterns.
-- The model shows a predisposition to false negatives for the unprivileged group (High School Graduates), meaning it tends to predict a budget below $300 for these users even when their actual budget exceeds that threshold. Conversely, the model shows low false positive rates overall but may slightly over-predict budgets for users with higher education credentials, which could lead to recommendations that exceed their actual spending capacity.
-- Further ethical AI analyses I would apply beyond this project: Given more time, I would implement intersectional fairness analysis to examine bias across combinations of protected attributes (e.g., education level AND gender AND age), apply adversarial debiasing as an in-processing technique to complement the current pre- and post-processing approaches, conduct user studies with diverse participants to validate that the model's recommendations are perceived as fair and useful, and implement continuous fairness monitoring in production to detect distribution shifts and emerging biases over time.
"""

**Optional**: You may choose to create a cohort analysis plot showing the fairness metric values on the y-axis and the Education Levels on the x-axis.

In [ ]:
# Optional: Fairness cohort analysis - balanced accuracy by education level
edu_fairness_data = {}
for level in education_levels:
    mask = edu_labels == level
    if mask.sum() > 0:
        subset_true = orig_test.labels.ravel()[mask]
        subset_pred = y_pred_mitigated[mask]
        if len(np.unique(subset_true)) > 1 and len(np.unique(subset_pred)) > 1:
            ba = balanced_accuracy_score(subset_true, subset_pred)
            edu_fairness_data[level] = ba

fig, ax = plt.subplots(figsize=(8, 5))
levels = list(edu_fairness_data.keys())
ba_values = list(edu_fairness_data.values())
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']
bars = ax.bar(levels, ba_values, color=colors[:len(levels)])
ax.set_xlabel('Education Level')
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Balanced Accuracy by Education Level (After Reweighing)')
ax.set_ylim(0.8, 1.0)
for bar, val in zip(bars, ba_values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('images/optional_fairness_cohort_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Articulate the ethical implications

Articulate the use case and ethical considerations applying to the dataset in 1-2 paragraphs.

**Hints:** 
 
- Think about the limitations of the dataset, potential biases that could be introduced into the use case, and the strengths and weaknesses of your ML model.
- Ethical Considerations:
 - Human-in-the-loop considerations: Can users control aspects of the model and inspect the features? If so, briefly describe how?
 - Describe the limitations and types of bias present in the data
 - Describe the failures of the ML model
 - It must include a section on any risk mitigation strategies you applied.
 - Describe Potential harms
 - It must include key contributing factors you found from your interpretability study, both before and after applying the bias mitigation strategy.

- Caveats and Recommendations
 - Potential lack of inclusiveness in the dataset
 - Predisposition of the model to false positives and/or negatives
 - It must also include 1-2 sentences on the further ethical AI analyses you would apply if given more time beyond this project.

- Business consequences
 - Potential positive impact of the IDOOU Budget Predicter AI
 - Reasons why users may lose trust in the application, and loss of revenue and brand reputation might occur to the organization

In [ ]:
business_consequences="""

-- Positive Impact: The IDOOU Budget Predictor AI has the potential to significantly enhance the user experience by automating the budget estimation process, enabling faster and more relevant activity recommendations. By reducing the friction of manually specifying budgets, users can focus on enjoying their activities, and the personalized approach can increase user engagement and satisfaction. Hotels and autonomous vehicle platforms that integrate IDOOU can offer a differentiated, intelligent concierge service that adds value for their customers and drives adoption of the platform.

-- Negative Impact: Users may lose trust in the IDOOU application if they perceive that the budget predictions are unfair or discriminatory, particularly if users with lower education credentials consistently receive recommendations for cheaper activities while users with higher education credentials are offered premium experiences. This perceived bias could lead to negative reviews, social media backlash, and regulatory scrutiny, resulting in loss of revenue and brand reputation damage.

-- Trust and Transparency: If users discover that their education level is a primary driver of budget predictions, they may feel that the app is making assumptions about their financial capability based on socioeconomic status, which could be seen as offensive and drive user churn across all demographic segments. Proactively disclosing the model's key factors and providing user override capabilities can mitigate this risk and build user confidence in the fairness of the system.
"""

In [ ]:
html_code = f"""
<html>
  <head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
      body {{
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        max-width: 1000px;
        margin: 0 auto;
        padding: 20px 40px;
        background-color: #f8f9fa;
        color: #2c3e50;
        line-height: 1.6;
      }}
      h1 {{
        background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%);
        color: white;
        padding: 25px;
        border-radius: 10px;
        text-align: center;
        margin-bottom: 30px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.1);
      }}
      h2 {{
        color: #2c3e50;
        border-bottom: 3px solid #3498db;
        padding-bottom: 8px;
        margin-top: 35px;
      }}
      .section {{
        background: white;
        padding: 20px 25px;
        border-radius: 8px;
        margin-bottom: 20px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05);
        border-left: 4px solid #3498db;
      }}
      .section-ethical {{
        border-left-color: #e74c3c;
      }}
      .section-metrics {{
        border-left-color: #27ae60;
      }}
      .section-business {{
        border-left-color: #9b59b6;
      }}
      .section-caveats {{
        border-left-color: #e67e22;
      }}
      img {{
        max-width: 100%;
        height: auto;
        border-radius: 6px;
        margin: 10px 0;
        box-shadow: 0 2px 8px rgba(0,0,0,0.1);
      }}
      .image-gallery {{
        display: flex;
        flex-direction: column;
        align-items: center;
        gap: 15px;
      }}
      .footer {{
        text-align: center;
        color: #7f8c8d;
        margin-top: 40px;
        padding: 15px;
        border-top: 1px solid #ddd;
        font-size: 0.9em;
      }}
    </style>
  </head>
  <body>
  <h1>Model Card - IDOOU AI Budget Predictor</h1>
  
  <h2>Model Details</h2>
  <div class="section">
  {model_details}
  </div>
  
  <h2>Intended Use</h2>
  <div class="section">
  {intended_use}
  </div>
  
  <h2>Factors</h2>
  <div class="section">
  {factors}
  </div>
  
  <h2>Metrics</h2>
  <div class="section section-metrics">
  {metrics}
  </div>
  
  <h2>Training Data</h2>
  <div class="section">
  {training_data}
  </div>
  
  <h2>Evaluation Data</h2>
  <div class="section">
  {eval_data}
  </div>
  
  <h2>Quantitative Analysis</h2>
  <div class="section section-metrics">
  {final_metrics_description}
  </div>
  
  <br/><b>Results of the AI model after applying the bias mitigation strategy</b><br/>
  
  <div class="image-gallery">
  {image_file_path}
  </div>

  <h2>Ethical Considerations</h2>
  <div class="section section-ethical">
  {ethical_considerations}
  </div>
  
  <h2>Caveats and Recommendations</h2>
  <div class="section section-caveats">
  {caveats_and_recommendations}
  </div>
  
  <h2>Business Consequences</h2>
  <div class="section section-business">
  {business_consequences}
  </div>
  
  <div class="footer">
    IDOOU Budget Predictor AI - Model Card | Generated as part of the AI Ethics Project
  </div>
  </body>
</html>"""
html_code = html_code.replace('--', '<br>--')

## Document the solution in a model card

You're at the finish line! Run the last few blocks of code to generate a simple html file with your model card content and the visualizations you generated for the final version of your model.

Make sure to open the html file and check that it is reflective of your model card content before submitting.

Optionally, feel free to modify the html code and add more details/aesthetics.

In [ ]:
with open('model_card.html', 'w') as f:
    f.write(html_code)